In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

clean_dir = Path('./jd_clean_output')
marts_dir = Path('./jd_marts_output')
marts_dir.mkdir(parents=True, exist_ok=True)

skus_ = pd.read_csv(clean_dir / 'skus_clean.csv', parse_dates=['activate_date', 'deactivate_date'], low_memory=False)
users_ = pd.read_csv(clean_dir / 'users_clean.csv', low_memory=False)
clicks_ = pd.read_csv(clean_dir / 'clicks_clean.csv', parse_dates=['request_time'], low_memory=False)
orders_ = pd.read_csv(clean_dir / 'orders_clean.csv', parse_dates=['order_time', 'order_date'], low_memory=False)
delivery_ = pd.read_csv(clean_dir / 'delivery_clean.csv', parse_dates=['ship_out_time', 'arr_station_time', 'arr_time'], low_memory=False)
inventory_ = pd.read_csv(clean_dir / 'inventory_clean.csv', parse_dates=['date'], low_memory=False)
network_ = pd.read_csv(clean_dir / 'network_clean.csv', low_memory=False)

In [2]:
def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def mode_or_first(s: pd.Series):
    s2 = s.dropna()
    if len(s2) == 0:
        return np.nan
    m = s2.mode()
    return m.iloc[0] if len(m) > 0 else s2.iloc[0]

In [3]:
sku_master = skus_.copy()

sku_master['is_1p'] = (sku_master['type'] == 1).astype('int8')
sku_master['is_3p'] = (sku_master['type'] == 2).astype('int8')

sku_master['attr1_missing'] = sku_master['attribute1'].isna().astype('int8')
sku_master['attr2_missing'] = sku_master['attribute2'].isna().astype('int8')

sku_master['attr1_placeholder'] = (sku_master['attribute1'].astype('string').str.strip() == '-').astype('int8')
sku_master['attr2_placeholder'] = (sku_master['attribute2'].astype('string').str.strip() == '-').astype('int8')

sku_master['any_attr_missing'] = (
    (sku_master['attr1_missing'] == 1) | (sku_master['attr2_missing'] == 1)
).astype('int8')

sku_master['any_attr_placeholder'] = (
    (sku_master['attr1_placeholder'] == 1) | (sku_master['attr2_placeholder'] == 1)
).astype('int8')

sku_master['is_activated_in_march'] = sku_master['activate_date'].notna().astype('int8')
sku_master['is_deactivated_in_march'] = sku_master['deactivate_date'].notna().astype('int8')

brand_sku_cnt = sku_master.groupby('brand_id', as_index=False).agg(
    brand_sku_cnt=('sku_id', 'nunique')
)
sku_master = sku_master.merge(brand_sku_cnt, on='brand_id', how='left')

display(sku_master.head())

,sku_id,type,brand_id,attribute1,attribute2,activate_date,deactivate_date,is_1p,is_3p,attr1_missing,attr2_missing,attr1_placeholder,attr2_placeholder,any_attr_missing,any_attr_placeholder,is_activated_in_march,is_deactivated_in_march,brand_sku_cnt
0,a234e08c57,1,c3ab4bf4d9,3.0,60.0,NaT,NaT,1,0,0,0,0,0,0,0,0,0,4
1,6449e1fd87,1,1d8b4b4c63,2.0,50.0,NaT,NaT,1,0,0,0,0,0,0,0,0,0,26
2,09b70fcd83,2,eb7d2a675a,3.0,70.0,NaT,NaT,0,1,0,0,0,0,0,0,0,0,35
3,acad9fed04,2,9b0d3a5fc6,3.0,70.0,NaT,NaT,0,1,0,0,0,0,0,0,0,0,831
4,2fa77e3b4d,2,b681299668,-,-,NaT,NaT,0,1,0,0,1,1,0,1,0,0,100


In [4]:
user_master = users_.copy()

user_master['is_plus'] = (user_master['plus'] == 1).astype('int8')
user_master['is_new_user'] = (user_master['user_level'] == -1).astype('int8')
user_master['is_enterprise_user'] = (user_master['user_level'] == 10).astype('int8')
user_master['is_tier12_city'] = user_master['city_level'].isin([1, 2]).astype('int8')
user_master['is_high_purchase_power'] = user_master['purchase_power'].isin([1, 2]).astype('int8')

demo_cols = ['gender', 'age', 'marital_status', 'education', 'purchase_power', 'city_level']
user_master['known_demo_flag'] = user_master[demo_cols].notna().any(axis=1).astype('int8')

display(user_master.head())

,user_id,user_level,first_order_month,plus,gender,age,marital_status,education,city_level,purchase_power,is_plus,is_new_user,is_enterprise_user,is_tier12_city,is_high_purchase_power,known_demo_flag
0,000089d6a6,1,2017-08,0,F,26-35,S,3,4,3,0,0,0,0,0,1
1,0000babd1f,1,2018-03,0,U,U,U,-1,-1,-1,0,0,0,0,0,1
2,0000bc018b,3,2016-06,0,F,>=56,M,3,2,3,0,0,0,1,0,1
3,0000d0e5ab,3,2014-06,0,M,26-35,M,3,2,2,0,0,0,1,1,1
4,0000dce472,3,2012-08,1,U,U,U,-1,-1,-1,1,0,0,0,0,1


In [5]:
discount_cols = [
    'direct_discount_per_unit',
    'quantity_discount_per_unit',
    'bundle_discount_per_unit',
    'coupon_discount_per_unit'
]

order_item_fact = orders_.copy()

order_item_fact['gmv'] = order_item_fact['quantity'].fillna(0) * order_item_fact['final_unit_price'].fillna(0)
order_item_fact['original_gmv'] = order_item_fact['quantity'].fillna(0) * order_item_fact['original_unit_price'].fillna(0)

for c in discount_cols:
    if c not in order_item_fact.columns:
        order_item_fact[c] = 0.0
    order_item_fact[c] = order_item_fact[c].fillna(0)

order_item_fact['unit_discount_total'] = (
    order_item_fact['direct_discount_per_unit']
    + order_item_fact['quantity_discount_per_unit']
    + order_item_fact['bundle_discount_per_unit']
    + order_item_fact['coupon_discount_per_unit']
)
order_item_fact['discount_amt'] = order_item_fact['unit_discount_total'] * order_item_fact['quantity'].fillna(0)
order_item_fact['discount_rate'] = safe_div(order_item_fact['unit_discount_total'], order_item_fact['original_unit_price'])

order_item_fact['gift_item'] = order_item_fact['gift_item'].fillna(0)
order_item_fact['gift_order_flag'] = (order_item_fact['gift_item'] == 1).astype('int8')
order_item_fact['cross_dc_flag'] = (order_item_fact['dc_ori'] != order_item_fact['dc_des']).astype('int8')
order_item_fact['same_or_next_day_promise_flag'] = (order_item_fact['promise'] == 1).astype('int8')

order_item_fact = (
    order_item_fact
    .merge(sku_master, on='sku_id', how='left', suffixes=('', '_sku'))
    .merge(user_master, on='user_id', how='left', suffixes=('', '_user'))
)

display(order_item_fact.head())

/tmp/ipykernel_3213/1330336395.py:4: RuntimeWarning: divide by zero encountered in divide
  return np.where((b == 0) | pd.isna(b), np.nan, a / b)
/tmp/ipykernel_3213/1330336395.py:4: RuntimeWarning: invalid value encountered in divide
  return np.where((b == 0) | pd.isna(b), np.nan, a / b)


,order_id,user_id,sku_id,order_date,order_time,quantity,type,promise,original_unit_price,final_unit_price,direct_discount_per_unit,quantity_discount_per_unit,bundle_discount_per_unit,coupon_discount_per_unit,gift_item,dc_ori,dc_des,discount_gap_check,discount_gap_abs,gmv,original_gmv,unit_discount_total,discount_amt,discount_rate,gift_order_flag,cross_dc_flag,same_or_next_day_promise_flag,type_sku,brand_id,attribute1,attribute2,activate_date,deactivate_date,is_1p,is_3p,attr1_missing,attr2_missing,attr1_placeholder,attr2_placeholder,any_attr_missing,any_attr_placeholder,is_activated_in_march,is_deactivated_in_march,brand_sku_cnt,user_level,first_order_month,plus,gender,age,marital_status,education,city_level,purchase_power,is_plus,is_new_user,is_enterprise_user,is_tier12_city,is_high_purchase_power,known_demo_flag
0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,2018-03-01,2018-03-01 17:14:25,1,2,-,89.0,79.0,0.0,10.0,0.0,0.0,0,4,28,0.0,0.0,79.0,89.0,10.0,10.0,0.112360,0,1,0,2.0,198cec62a1,4.0,100.0,NaT,NaT,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1231.0,1,2013-06,0,F,36-45,M,3,2,2,0,0,0,1,1,1
1,7444318d01,33a9e56257,067b673f2b,2018-03-01,2018-03-01 11:10:40,1,1,2,99.9,53.9,5.0,41.0,0.0,0.0,0,28,28,0.0,0.0,53.9,99.9,46.0,46.0,0.460460,0,0,0,1.0,9b0d3a5fc6,3.0,60.0,NaT,NaT,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,831.0,1,2016-05,0,F,26-35,S,2,2,3,0,0,0,1,0,1
2,f973b01694,4ea3cf408f,623d0a582a,2018-03-01,2018-03-01 09:13:26,1,1,2,78.0,58.5,19.5,0.0,0.0,0.0,0,28,28,0.0,0.0,58.5,78.0,19.5,19.5,0.250000,0,0,0,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2013-02,0,F,16-25,S,2,2,2,0,0,0,1,1,1
3,8c1cec8d4b,b87cb736cb,fc5289b139,2018-03-01,2018-03-01 21:29:50,1,1,2,61.0,35.0,0.0,26.0,0.0,0.0,0,4,28,0.0,0.0,35.0,61.0,26.0,26.0,0.426230,0,1,0,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,2014-08,0,F,26-35,M,3,2,2,0,0,0,1,1,1
4,d43a33c38a,4829223b6f,623d0a582a,2018-03-01,2018-03-01 19:13:37,1,1,1,78.0,53.0,19.0,0.0,0.0,6.0,0,3,16,0.0,0.0,53.0,78.0,25.0,25.0,0.320513,0,1,0,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2014-10,0,F,26-35,M,2,2,4,0,0,0,1,0,1


In [6]:
click_intent_fact = clicks_.copy()

click_intent_fact['click_date'] = click_intent_fact['request_time'].dt.floor('D')
click_intent_fact['hour'] = click_intent_fact['request_time'].dt.hour
click_intent_fact['dow'] = click_intent_fact['request_time'].dt.dayofweek
click_intent_fact['is_weekend'] = click_intent_fact['dow'].isin([5, 6]).astype('int8')

click_intent_fact['hour_bucket'] = pd.cut(
    click_intent_fact['hour'],
    bins=[-1, 5, 9, 12, 17, 21, 23],
    labels=['overnight', 'morning', 'late_morning', 'afternoon', 'evening', 'late_night']
)

click_intent_fact = (
    click_intent_fact
    .groupby(['user_id', 'sku_id', 'channel', 'click_date'], observed=True)
    .agg(
        first_click_ts=('request_time', 'min'),
        last_click_ts=('request_time', 'max'),
        click_cnt=('request_time', 'size'),
        hour=('hour', 'first'),
        dow=('dow', 'first'),
        is_weekend=('is_weekend', 'first'),
        hour_bucket=('hour_bucket', 'first')
    )
    .reset_index()
)

click_intent_fact['is_known_user'] = click_intent_fact['user_id'].isin(user_master['user_id']).astype('int8')

click_intent_fact = click_intent_fact.merge(
    sku_master[
        ['sku_id', 'type', 'brand_id', 'attribute1', 'attribute2',
         'any_attr_missing', 'any_attr_placeholder', 'is_1p', 'is_3p']
    ],
    on='sku_id',
    how='left'
)

display(click_intent_fact.head())

,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,is_known_user,type,brand_id,attribute1,attribute2,any_attr_missing,any_attr_placeholder,is_1p,is_3p
0,-,00078c2a0f,mobile,2018-03-07,2018-03-07 09:52:59,2018-03-07 09:52:59,1,9.0,2.0,0,morning,0,2,4240d14733,3.0,60.0,0,0,0,1
1,-,00078c2a0f,pc,2018-03-06,2018-03-06 09:46:41,2018-03-06 09:46:41,1,9.0,1.0,0,morning,0,2,4240d14733,3.0,60.0,0,0,0,1
2,-,0009ac56b7,pc,2018-03-07,2018-03-07 03:44:59,2018-03-07 13:27:17,2,13.0,2.0,0,afternoon,0,2,9b0d3a5fc6,3.0,60.0,0,0,0,1
3,-,000aa92b82,mobile,2018-03-05,2018-03-05 16:37:50,2018-03-05 16:37:50,1,16.0,0.0,0,afternoon,0,2,9d3465eacc,4.0,100.0,0,0,0,1
4,-,000aa92b82,mobile,2018-03-08,2018-03-08 11:43:59,2018-03-08 11:43:59,1,11.0,3.0,0,late_morning,0,2,9d3465eacc,4.0,100.0,0,0,0,1


In [7]:
order_header = (
    order_item_fact
    .groupby('order_id', as_index=False)
    .agg(
        user_id=('user_id', 'first'),
        order_time=('order_time', 'min'),
        order_date=('order_date', 'min'),
        order_line_cnt=('sku_id', 'size'),
        distinct_sku_cnt=('sku_id', 'nunique'),
        total_qty=('quantity', 'sum'),
        type=('type', mode_or_first),
        promise=('promise', mode_or_first),
        dc_ori=('dc_ori', mode_or_first),
        dc_des=('dc_des', mode_or_first),
    )
)

order_gmv = (
    order_item_fact
    .groupby('order_id', as_index=False)
    .agg(
        total_gmv=('gmv', 'sum'),
        total_original_gmv=('original_gmv', 'sum'),
        total_discount_amt=('discount_amt', 'sum')
    )
)

order_header = order_header.merge(order_gmv, on='order_id', how='left')
order_header['cross_dc_flag'] = (order_header['dc_ori'] != order_header['dc_des']).astype('int8')

display(order_header.head())

,order_id,user_id,order_time,order_date,order_line_cnt,distinct_sku_cnt,total_qty,type,promise,dc_ori,dc_des,total_gmv,total_original_gmv,total_discount_amt,cross_dc_flag
0,0000095025,57648ed1fc,2018-03-19 11:11:34,2018-03-19,1,1,1,1,1,2,2,176.2,230.0,53.8,0
1,00000e13eb,c113527e40,2018-03-09 12:40:42,2018-03-09,1,1,1,2,2,9,36,56.0,56.0,0.0,1
2,0000132b39,c4f5626c0d,2018-03-13 16:30:35,2018-03-13,1,1,1,1,2,3,16,85.0,89.0,4.0,1
3,000038f581,893d65afc9,2018-03-12 13:54:49,2018-03-12,1,1,1,2,2,2,6,55.0,158.0,103.0,1
4,000064fa67,99439045cb,2018-03-02 10:56:17,2018-03-02,2,2,2,1,1,27,27,208.0,298.0,90.0,0


In [8]:
delivery_fact = delivery_.merge(
    order_header,
    on='order_id',
    how='left',
    suffixes=('', '_order')
)

delivery_fact['ship_hours'] = (delivery_fact['ship_out_time'] - delivery_fact['order_time']).dt.total_seconds() / 3600
delivery_fact['station_hours'] = (delivery_fact['arr_station_time'] - delivery_fact['ship_out_time']).dt.total_seconds() / 3600
delivery_fact['last_mile_hours'] = (delivery_fact['arr_time'] - delivery_fact['arr_station_time']).dt.total_seconds() / 3600
delivery_fact['delivery_total_hours'] = (delivery_fact['arr_time'] - delivery_fact['ship_out_time']).dt.total_seconds() / 3600

package_cnt = delivery_fact.groupby('order_id', as_index=False).agg(
    package_cnt=('package_id', 'nunique')
)
delivery_fact = delivery_fact.merge(package_cnt, on='order_id', how='left')
delivery_fact['split_order_flag'] = (delivery_fact['package_cnt'] > 1).astype('int8')

display(delivery_fact.head())

,package_id,order_id,type,ship_out_time,arr_station_time,arr_time,user_id,order_time,order_date,order_line_cnt,distinct_sku_cnt,total_qty,type_order,promise,dc_ori,dc_des,total_gmv,total_original_gmv,total_discount_amt,cross_dc_flag,ship_hours,station_hours,last_mile_hours,delivery_total_hours,package_cnt,split_order_flag
0,dc3d6d2258,dc3d6d2258,1,2018-03-01 08:00:00,2018-03-01 15:00:00,2018-03-01 18:00:00,ee666e25c3,2018-03-01 06:21:07,2018-03-01,1,1,1,1,1,4,4,33.0,36.0,3.0,0,1.648056,7.0,3.0,10.0,1,0
1,19802a570c,19802a570c,1,2018-03-01 10:00:00,2018-03-01 15:00:00,2018-03-01 17:00:00,845df5b5f2,2018-03-01 09:10:09,2018-03-01,1,1,1,1,1,2,2,59.0,78.0,19.0,0,0.830833,5.0,2.0,7.0,1,0
2,e22627af66,e22627af66,1,2018-03-01 11:00:00,2018-03-01 15:00:00,2018-03-01 17:00:00,cae0d8c01f,2018-03-01 10:50:41,2018-03-01,7,2,13,1,1,5,5,186.0,230.0,44.0,0,0.155278,4.0,2.0,6.0,1,0
3,50d11a586d,50d11a586d,1,2018-03-01 10:00:00,2018-03-01 16:00:00,2018-03-01 19:00:00,3aaeaef748,2018-03-01 08:50:19,2018-03-01,1,1,1,1,1,5,5,58.5,78.0,19.5,0,1.161389,6.0,3.0,9.0,1,0
4,a3bfe38bf4,a3bfe38bf4,1,2018-03-01 11:00:00,2018-03-01 16:00:00,2018-03-01 17:00:00,709961b53e,2018-03-01 09:17:20,2018-03-01,2,2,2,1,1,2,2,115.5,156.0,40.5,0,1.711111,5.0,1.0,6.0,1,0


In [9]:
inventory_network_fact = inventory_.merge(network_, on='dc_id', how='left')
display(inventory_network_fact.head())

,dc_id,sku_id,date,region_id
0,9,50f6f91962,2018-03-01,9
1,9,7f0ddbcdde,2018-03-01,9
2,9,8ad5789d74,2018-03-01,9
3,9,468d34eda4,2018-03-01,9
4,9,460afaddb6,2018-03-01,9


In [10]:
sku_master.to_csv(marts_dir / 'sku_master.csv', index=False)
user_master.to_csv(marts_dir / 'user_master.csv', index=False)
order_item_fact.to_csv(marts_dir / 'order_item_fact.csv', index=False)
click_intent_fact.to_csv(marts_dir / 'click_intent_fact.csv', index=False)
order_header.to_csv(marts_dir / 'order_header.csv', index=False)
delivery_fact.to_csv(marts_dir / 'delivery_fact.csv', index=False)
inventory_network_fact.to_csv(marts_dir / 'inventory_network_fact.csv', index=False)

print('已导出到:', marts_dir.resolve())

已导出到: /home/houga/JD/jd_marts_output
